# 🎬 CineMatch AI


## Phase 3 - Recommendation Model

### 1. Import Libraries

In [1]:
import pandas as pd
import numpy as np

from sklearn.feature_extraction.text import CountVectorizer
from sklearn.metrics.pairwise import cosine_similarity

import nltk
from nltk.stem.porter import PorterStemmer

import pickle

### 2. Creating Stemmer

In [2]:
ps = PorterStemmer()

### 3. Loading Processed Dataset

In [3]:
new_df = pd.read_csv("../data/processed/movies_processed.csv")

new_df.head()

,movie_id,title,tags
0,19995,Avatar,"in the 22nd century, a paraplegic marine is di..."
1,285,Pirates of the Caribbean: At World's End,"captain barbossa, long believed to be dead, ha..."
2,206647,Spectre,a cryptic message from bond’s past sends him o...
3,49026,The Dark Knight Rises,following the death of district attorney harve...
4,49529,John Carter,"john carter is a war-weary, former military ca..."


### 4. Dataset Shape

In [5]:
new_df.shape

(4806, 3)

### 5. Streaming Fxn

In [6]:
def stem(text):
    y = []

    for i in text.split():
        y.append(ps.stem(i))

    return " ".join(y)

### 6. Applying Stemming

In [7]:
new_df["tags"] = new_df["tags"].apply(stem)

### 7. Verifying

In [8]:
new_df.head()

,movie_id,title,tags
0,19995,Avatar,"in the 22nd century, a parapleg marin is dispa..."
1,285,Pirates of the Caribbean: At World's End,"captain barbossa, long believ to be dead, ha c..."
2,206647,Spectre,a cryptic messag from bond’ past send him on a...
3,49026,The Dark Knight Rises,follow the death of district attorney harvey d...
4,49529,John Carter,"john carter is a war-weary, former militari ca..."


### 8. Creating CountVectorizer

In [9]:
cv = CountVectorizer(
    max_features=5000,
    stop_words="english"
)

### 9. Converting Tags to Vectors

In [10]:
vectors = cv.fit_transform(new_df["tags"]).toarray()

### 10. Checking Shape

In [11]:
vectors.shape

(4806, 5000)

### 11. Cosine similarity

In [12]:
similarity = cosine_similarity(vectors)

### 12. Verifying 

In [13]:
similarity.shape

(4806, 4806)

### 13. Recommendation Fxn

In [14]:
def recommend(movie):

    movie = movie.lower()

    try:
        movie_index = new_df[
            new_df["title"].str.lower() == movie
        ].index[0]

    except IndexError:
        print("Movie not found.")
        return

    distances = similarity[movie_index]

    movie_list = sorted(
        list(enumerate(distances)),
        reverse=True,
        key=lambda x: x[1]
    )[1:6]

    print(f"\nRecommendations for '{movie.title()}':\n")

    for i in movie_list:
        print(new_df.iloc[i[0]].title)

### 14. Testing

In [16]:
recommend("Avatar")


Recommendations for 'Avatar':

Aliens vs Predator: Requiem
Aliens
Falcon Rising
Independence Day
Titan A.E.


In [17]:
recommend("Batman Begins")


Recommendations for 'Batman Begins':

The Dark Knight
Batman
Batman
The Dark Knight Rises
10th & Wolf


### 15. Saving the Model

In [18]:
pickle.dump(
    new_df,
    open("../models/movies.pkl", "wb")
)

### 16. Saving similarity Matrix

In [19]:
pickle.dump(
    similarity,
    open("../models/similarity.pkl", "wb")
)

### 17. Verification

In [20]:
import os

print(os.path.exists("../models/movies.pkl"))
print(os.path.exists("../models/similarity.pkl"))

True
True


### 18. Final Test

In [21]:
recommend("The Dark Knight")
recommend("Iron Man")
recommend("Avatar")
recommend("Spectre")


Recommendations for 'The Dark Knight':

The Dark Knight Rises
Batman Begins
Batman Returns
Batman Forever
Batman

Recommendations for 'Iron Man':

Iron Man 3
Iron Man 2
Avengers: Age of Ultron
The Avengers
Captain America: Civil War

Recommendations for 'Avatar':

Aliens vs Predator: Requiem
Aliens
Falcon Rising
Independence Day
Titan A.E.

Recommendations for 'Spectre':

Quantum of Solace
Skyfall
Never Say Never Again
From Russia with Love
Octopussy
